In [ ]:
!pip install transformers

In [ ]:
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: fineGrained).
The token `magang5` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-crede

In [ ]:
import torch
import torch
import pandas as pd
from tqdm import tqdm

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained("willywonka19/indobert-classification-rs-3")
tokenizer = BertTokenizer.from_pretrained("indobenchmark/indobert-base-p1")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/885 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

In [ ]:
df = pd.read_csv("/content/result_1.csv")

In [ ]:
labels = ['pelayanan', 'fasilitas']

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_batch(texts, batch_size=8):
    model.eval()
    results = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]

        # Tokenisasi & kirim ke device
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        # Prediksi
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()  # probabilitas [0,1]

        # Simpan probabilitas + label prediksi tertinggi
        for prob_row in probs:
            result = {label: float(p) for label, p in zip(labels, prob_row)}
            result["predicted_labels"] = [label for label, p in result.items() if label in labels and p >= 0.5]
            results.append(result)

    return pd.DataFrame(results)

In [ ]:
hasil_label = predict_batch(df["text"].tolist())

  0%|          | 0/955 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
100%|██████████| 955/955 [00:41<00:00, 23.17it/s]


In [ ]:
hasil_label

,pelayanan,fasilitas,predicted_labels
0,0.996950,0.989263,"[pelayanan, fasilitas]"
1,0.997084,0.988736,"[pelayanan, fasilitas]"
2,0.997248,0.004785,[pelayanan]
3,0.998670,0.012462,[pelayanan]
4,0.994812,0.991859,"[pelayanan, fasilitas]"
...,...,...,...
7634,0.998652,0.013547,[pelayanan]
7635,0.998214,0.969155,"[pelayanan, fasilitas]"
7636,0.154345,0.233327,[]
7637,0.122932,0.100288,[]


In [ ]:
df_combined = pd.concat([df, hasil_label], axis=1)
display(df_combined.head())

,title,stars,text,bin_pelayanan,bin_fasilitas,predicted_labels_1,labels,pelayanan,fasilitas,predicted_labels
0,RS Brayat Minulya,5,telah dirawat di sini selama beberapa hari di ...,1,1,"['pelayanan', 'fasilitas']",positive,0.996950,0.989263,"[pelayanan, fasilitas]"
1,RS Brayat Minulya,5,bersih dan rapi petugas peeawatdokter ditanya ...,1,1,"['pelayanan', 'fasilitas']",positive,0.997084,0.988736,"[pelayanan, fasilitas]"
2,RS Brayat Minulya,5,terimakasih atas pelayanan dan perawatan ibu s...,1,0,['pelayanan'],positive,0.997248,0.004785,[pelayanan]
3,RS Brayat Minulya,5,pelayanan diruang yosef bagus perawat ramah ra...,1,0,['pelayanan'],positive,0.998670,0.012462,[pelayanan]
4,RS Brayat Minulya,5,tempatnya bagusrapi nyaman sekali ruangan nya ...,1,1,"['pelayanan', 'fasilitas']",positive,0.994812,0.991859,"[pelayanan, fasilitas]"


In [ ]:
# Drop the 'predicted_labels' column (as it is duplicated)
df_combined = df_combined.drop('predicted_labels', axis=1)
df_combined

,title,stars,text,bin_pelayanan,bin_fasilitas,predicted_labels_1,labels,pelayanan,fasilitas
0,RS Brayat Minulya,5,telah dirawat di sini selama beberapa hari di ...,1,1,"['pelayanan', 'fasilitas']",positive,0.996950,0.989263
1,RS Brayat Minulya,5,bersih dan rapi petugas peeawatdokter ditanya ...,1,1,"['pelayanan', 'fasilitas']",positive,0.997084,0.988736
2,RS Brayat Minulya,5,terimakasih atas pelayanan dan perawatan ibu s...,1,0,['pelayanan'],positive,0.997248,0.004785
3,RS Brayat Minulya,5,pelayanan diruang yosef bagus perawat ramah ra...,1,0,['pelayanan'],positive,0.998670,0.012462
4,RS Brayat Minulya,5,tempatnya bagusrapi nyaman sekali ruangan nya ...,1,1,"['pelayanan', 'fasilitas']",positive,0.994812,0.991859
...,...,...,...,...,...,...,...,...,...
7634,Rumah Sakit Umum Pusat Surakarta,4,lama banget pelayanan untuk pasien baru ibu ny...,1,0,['pelayanan'],negative,0.998652,0.013547
7635,Rumah Sakit Umum Pusat Surakarta,4,hospital yg sangat rapi pelayanan bagus,1,1,"['pelayanan', 'fasilitas']",positive,0.998214,0.969155
7636,Rumah Sakit Umum Pusat Surakarta,4,khusus paru,0,0,[],positive,0.154345,0.233327
7637,Rumah Sakit Umum Pusat Surakarta,5,rs paru,0,0,[],neutral,0.122932,0.100288


In [ ]:
df_combined = df_combined.rename(columns={'fasilitas': 'prob_fasilitas', 'pelayanan': 'prob_layanan'})
display(df_combined.head())

,title,stars,text,bin_pelayanan,bin_fasilitas,predicted_labels_1,labels,prob_layanan,prob_fasilitas
0,RS Brayat Minulya,5,telah dirawat di sini selama beberapa hari di ...,1,1,"['pelayanan', 'fasilitas']",positive,0.996950,0.989263
1,RS Brayat Minulya,5,bersih dan rapi petugas peeawatdokter ditanya ...,1,1,"['pelayanan', 'fasilitas']",positive,0.997084,0.988736
2,RS Brayat Minulya,5,terimakasih atas pelayanan dan perawatan ibu s...,1,0,['pelayanan'],positive,0.997248,0.004785
3,RS Brayat Minulya,5,pelayanan diruang yosef bagus perawat ramah ra...,1,0,['pelayanan'],positive,0.998670,0.012462
4,RS Brayat Minulya,5,tempatnya bagusrapi nyaman sekali ruangan nya ...,1,1,"['pelayanan', 'fasilitas']",positive,0.994812,0.991859


In [ ]:
# simpan
df_combined.to_csv("result.csv", index=False)